In [1]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 1. Paths
INPUT_PATH = Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\interim\delhi_15min_density.parquet")
GRAPH_OUT_PATH = Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\intersection_graph.npz")

print("Loading 15-minute multi-approach dataset...")
df = pd.read_parquet(INPUT_PATH)

# Nodes: The three intersection approaches
nodes = ["Approach1_Queue", "Approach2_Queue", "Approach3_Queue"]
num_nodes = len(nodes)

Loading 15-minute multi-approach dataset...


In [3]:
# 2. Construct Data-Driven Adjacency Matrix via Cross-Correlation
print("1. Computing spatial correlation adjacency matrix...")
corr_matrix = df[nodes].corr().values

# Apply thresholding: keep self-loops and positive correlations
adj_matrix = np.where(corr_matrix > 0.1, corr_matrix, 0.0)
np.fill_diagonal(adj_matrix, 1.0) # Self-loops

print("Adjacency Matrix (3x3):")
print(np.round(adj_matrix, 4))

1. Computing spatial correlation adjacency matrix...
Adjacency Matrix (3x3):
[[1.     0.6454 0.4405]
 [0.6454 1.     0.4795]
 [0.4405 0.4795 1.    ]]


In [4]:
# 3. Create Multi-Node Feature Arrays
# Features per node: [Queue_Density, Stop_Density]
features_per_node = [
    ["Approach1_Queue", "Approach1_Stop"],
    ["Approach2_Queue", "Approach2_Stop"],
    ["Approach3_Queue", "Approach3_Stop"]
]

# Shape: [Total_Timesteps, Nodes, Features]
data_all = np.zeros((len(df), num_nodes, 2))
scalers = {}

for n_idx, (q_col, s_col) in enumerate(features_per_node):
    scaler = MinMaxScaler()
    scaled_feats = scaler.fit_transform(df[[q_col, s_col]])
    data_all[:, n_idx, :] = scaled_feats
    scalers[n_idx] = scaler

In [5]:
# 4. Generate Spatiotemporal 4D Sliding Windows: [Samples, Time_Window, Nodes, Features]
def create_spatiotemporal_windows(data, window_size=4, horizon=1):
    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i : i + window_size, :, :])
        # Predict Queue Density (feature 0) across all 3 nodes at t + horizon
        y.append(data[i + window_size + horizon - 1, :, 0])
    return np.array(X), np.array(y)

window_size = 4  # 1 hour lookback (4 * 15m)
X_st, y_st = create_spatiotemporal_windows(data_all, window_size=window_size, horizon=1)

In [6]:
# 5. Chronological Train/Test Split (70/30)
split_idx = int(len(X_st) * 0.7)
X_train, X_test = X_st[:split_idx], X_st[split_idx:]
y_train, y_test = y_st[:split_idx], y_st[split_idx:]

print(f"\n--- Spatiotemporal Data Summary ---")
print(f"X_train tensor shape : {X_train.shape}  (Samples, Timesteps, Nodes, Features)")
print(f"y_train tensor shape : {y_train.shape}  (Samples, Nodes)")
print(f"X_test tensor shape  : {X_test.shape}")
print(f"y_test tensor shape  : {y_test.shape}")


--- Spatiotemporal Data Summary ---
X_train tensor shape : (1677, 4, 3, 2)  (Samples, Timesteps, Nodes, Features)
y_train tensor shape : (1677, 3)  (Samples, Nodes)
X_test tensor shape  : (719, 4, 3, 2)
y_test tensor shape  : (719, 3)


In [7]:
# 6. Save Graph Structure and Preprocessed Tensors
GRAPH_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    GRAPH_OUT_PATH,
    adj=adj_matrix,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)
print(f"[SUCCESS] Saved graph structure and tensors to: {GRAPH_OUT_PATH}")

[SUCCESS] Saved graph structure and tensors to: C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\intersection_graph.npz
